In [2]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1193").setMaster("local[4]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

25/08/07 00:52:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
'''
Table: Transactions

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| id            | int     |
| country       | varchar |
| state         | enum    |
| amount        | int     |
| trans_date    | date    |
+---------------+---------+
id is the primary key of this table.
The table has information about incoming transactions.
The state column is an enum of type ["approved", "declined"].
 

Write an SQL query to find for each month and country, the number of transactions and their total amount, the number of 
approved transactions and their total amount.

Return the result table in any order.

The query result format is in the following example.

 

Example 1:

Input: 
Transactions table:
+------+---------+----------+--------+------------+
| id   | country | state    | amount | trans_date |
+------+---------+----------+--------+------------+
| 121  | US      | approved | 1000   | 2018-12-18 |
| 122  | US      | declined | 2000   | 2018-12-19 |
| 123  | US      | approved | 2000   | 2019-01-01 |
| 124  | DE      | approved | 2000   | 2019-01-07 |
+------+---------+----------+--------+------------+
Output: 
+----------+---------+-------------+----------------+--------------------+-----------------------+
| month    | country | trans_count | approved_count | trans_total_amount | approved_total_amount |
+----------+---------+-------------+----------------+--------------------+-----------------------+
| 2018-12  | US      | 2           | 1              | 3000               | 1000                  |
| 2019-01  | US      | 1           | 1              | 2000               | 2000                  |
| 2019-01  | DE      | 1           | 1              | 2000               | 2000                  |
+----------+---------+-------------+----------------+--------------------+-----------------------+
'''

In [3]:
data = [
(121,'US','approved',1000,'2018-12-18'),
(122,'US','declined',2000,'2018-12-19'),
(123,'US','approved',2000,'2019-01-01'),
(124,'DE','approved',2000,'2019-01-07')    
]
schema = ['id','country','state','amount','trans_date']

In [4]:
df = spark.createDataFrame(data=data, schema=schema)
df.show()

+---+-------+--------+------+----------+
| id|country|   state|amount|trans_date|
+---+-------+--------+------+----------+
|121|     US|approved|  1000|2018-12-18|
|122|     US|declined|  2000|2018-12-19|
|123|     US|approved|  2000|2019-01-01|
|124|     DE|approved|  2000|2019-01-07|
+---+-------+--------+------+----------+



In [21]:
df.select(
            F.concat_ws('-',F.year(F.col("trans_date")),F.month(F.col("trans_date"))).alias("month"),
            F.col("country"),
            F.col("state"),
            F.col("amount")
         )\
    .groupBy(F.col("month"),F.col("country"))\
    .agg( 
        F.count(F.col("*")).alias("trans_count"),
        F.sum(F.when(F.col("state") == "approved", 1 ).otherwise(0)).alias("approved_count"),
        F.sum(F.col("amount")).alias("trans_total_amount"),
        F.sum(F.when(F.col("state") == 'approved', F.col("amount")).otherwise(0)).alias("approved_total_amount")
    )\
    .show(truncate = False)




+-------+-------+-----------+--------------+------------------+---------------------+
|month  |country|trans_count|approved_count|trans_total_amount|approved_total_amount|
+-------+-------+-----------+--------------+------------------+---------------------+
|2018-12|US     |2          |1             |3000              |1000                 |
|2019-1 |US     |1          |1             |2000              |2000                 |
|2019-1 |DE     |1          |1             |2000              |2000                 |
+-------+-------+-----------+--------------+------------------+---------------------+



## Melwin Note 
## Remember when ever you are having a select statement before the groupBy and aggregate <br> make sure all the columns which is required for the aggregate is mentioned in the select statement
## See the above example

## SQL Solution
<pre>
SELECT CONCAT(YEAR(trans_date),'-'.MONTH(trans_date)) as month, country, count(*) as trans_count, 
       SUM(CASE WHEN state = 'approved' THEN 1 ELSE 0 END) as  approved_count,
       SUM(amount) as trans_total_amount,
       SUM(CASE WHEN state = 'approved' THEN amount ELSE 0 END) as approved_total_amount
FROM Transactions
GROUP BY month, country
</pre>